# LHCO CWoLa Anomaly Detection Notebook

This notebook builds a **Classification Without Labels (CWoLa)** pipeline on the **LHC Olympics 2020 R&D dataset** using the large Zenodo HDF5 file.

## Workflow
1. Set up environment
2. Download the big LHCO dataset from Zenodo
3. Inspect and load the HDF5 table
4. Build physics features
5. Study the dijet mass distribution
6. Define signal region and sidebands
7. Build the CWoLa training dataset
8. Train a classifier excluding `mjj`
9. Evaluate on pseudo-labels and truth labels
10. Produce plots and candidate-event tables




In [ ]:
# ============================================
# SECTION 1: IMPORTS AND ENVIRONMENT SETUP
# ============================================
import os
import json
import math
import requests
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h5py

from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve, roc_auc_score, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 20)

print("TensorFlow version:", tf.__version__)
print("Setup complete.")


In [ ]:
# ============================================
# SECTION 2: CREATE PROJECT FOLDERS
# ============================================
PROJECT_DIR = Path("/content/lhco_cwola_project")
DATA_DIR = PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
FIG_DIR = PROJECT_DIR / "figures"
MODEL_DIR = PROJECT_DIR / "models"
TABLE_DIR = PROJECT_DIR / "tables"

for folder in [PROJECT_DIR, DATA_DIR, RAW_DIR, PROCESSED_DIR, FIG_DIR, MODEL_DIR, TABLE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project directory :", PROJECT_DIR)
print("Raw data directory:", RAW_DIR)
print("Processed dir     :", PROCESSED_DIR)
print("Figures dir       :", FIG_DIR)
print("Models dir        :", MODEL_DIR)
print("Tables dir        :", TABLE_DIR)


## Dataset download

This section downloads the **large 2.8 GB** LHCO file directly from the Zenodo record shown earlier.  
If the file already exists in Colab storage, the downloader skips it.


In [ ]:
# ============================================
# SECTION 3: DOWNLOAD THE BIG LHCO DATASET
# ============================================
ZENODO_RECORD_ID = "4536377"
ZENODO_API_URL = f"https://zenodo.org/api/records/{ZENODO_RECORD_ID}"

TARGET_FILENAME = "events_anomalydetection.h5"
OUTPUT_PATH = RAW_DIR / TARGET_FILENAME

def fetch_zenodo_metadata(api_url):
    response = requests.get(api_url, timeout=60)
    response.raise_for_status()
    return response.json()

def extract_files_from_metadata(metadata):
    files = []
    for item in metadata.get("files", []):
        filename = item.get("key", "unknown_file")
        size = item.get("size", None)
        links = item.get("links", {})
        direct_url = (
            links.get("self")
            or links.get("download")
            or item.get("self")
            or item.get("download")
        )
        files.append({
            "filename": filename,
            "size": size,
            "url": direct_url
        })
    return files

def human_readable_size(num_bytes):
    if num_bytes is None:
        return "unknown"
    units = ["B", "KB", "MB", "GB", "TB"]
    size = float(num_bytes)
    for unit in units:
        if size < 1024 or unit == units[-1]:
            return f"{size:.2f} {unit}"
        size /= 1024
    return f"{num_bytes} B"

def download_file(url, output_path, expected_size=None, chunk_size=1024 * 1024):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with requests.get(url, stream=True, timeout=60) as response:
        response.raise_for_status()

        total = expected_size
        if total is None:
            content_length = response.headers.get("Content-Length")
            if content_length is not None:
                total = int(content_length)

        with open(output_path, "wb") as f, tqdm(
            total=total,
            unit="B",
            unit_scale=True,
            unit_divisor=1024,
            desc=output_path.name
        ) as pbar:
            for chunk in response.iter_content(chunk_size=chunk_size):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))

    print(f"Download complete: {output_path}")

metadata = fetch_zenodo_metadata(ZENODO_API_URL)
files = extract_files_from_metadata(metadata)

print("Available files in record:")
for i, f in enumerate(files, 1):
    print(f"{i:2d}. {f['filename']:<45} {human_readable_size(f['size'])}")

selected_file = None
for f in files:
    if f["filename"] == TARGET_FILENAME:
        selected_file = f
        break

if selected_file is None:
    raise ValueError(f"Could not find {TARGET_FILENAME} in Zenodo record.")

print("\nSelected file:")
print("Filename:", selected_file["filename"])
print("Size    :", human_readable_size(selected_file["size"]))
print("URL     :", selected_file["url"])

if OUTPUT_PATH.exists():
    print(f"\nFile already exists at: {OUTPUT_PATH}")
    print("Existing size:", human_readable_size(OUTPUT_PATH.stat().st_size))
else:
    download_file(selected_file["url"], OUTPUT_PATH, expected_size=selected_file["size"])


## HDF5 inspection and loading

The large file is stored as a pandas HDF table under the group `df`.


In [ ]:
# ============================================
# SECTION 4: SAFE INSPECTION OF BIG HDF5 FILE
# ============================================
print("File exists:", OUTPUT_PATH.exists())
print("Path       :", OUTPUT_PATH)
print("Size       :", human_readable_size(OUTPUT_PATH.stat().st_size))

def inspect_hdf5_structure(h5_path):
    print(f"\nInspecting HDF5 structure for:\n{h5_path}\n")
    with h5py.File(h5_path, "r") as f:
        def visitor(name, obj):
            if isinstance(obj, h5py.Group):
                print(f"GROUP   : {name}")
            elif isinstance(obj, h5py.Dataset):
                print(f"DATASET : {name} | shape={obj.shape} | dtype={obj.dtype}")
        f.visititems(visitor)

inspect_hdf5_structure(OUTPUT_PATH)


In [ ]:
# ============================================
# SECTION 5: CHECK IF IT IS A PANDAS HDF STORE
# ============================================
try:
    store = pd.HDFStore(str(OUTPUT_PATH), mode="r")
    print("Pandas HDF keys found:")
    print(store.keys())
    store.close()
except Exception as e:
    print("Could not open as pandas HDFStore.")
    print("Reason:", e)


In [ ]:
# ============================================
# SECTION 6: LOAD PREVIEW AND FULL DATAFRAME
# ============================================
HDF_KEY = "df"

df_preview = pd.read_hdf(OUTPUT_PATH, key=HDF_KEY, stop=5)
print("Preview shape:", df_preview.shape)
display(df_preview)
print("\nColumns:")
print(df_preview.columns.tolist())

df = pd.read_hdf(OUTPUT_PATH, key=HDF_KEY)
print("\nFull dataframe loaded.")
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())


In [ ]:
# ============================================
# SECTION 7: BASIC SANITY CHECKS
# ============================================
print("First 5 rows:")
display(df.head())

print("\nMissing values per column:")
print(df.isnull().sum())

print("\nDtypes:")
print(df.dtypes)

if "label" in df.columns:
    print("\nLabel distribution:")
    print(df["label"].value_counts(dropna=False))
    print("\nSignal fraction:", df["label"].mean())
else:
    print("\nNo 'label' column found.")


## Physics feature engineering

We compute:
- jet transverse momenta
- jet energies
- dijet invariant mass `mjj`
- tau ratios `tau21` and `tau32`
- balance and mass-ratio features


In [ ]:
# ============================================
# SECTION 8: PHYSICS FEATURE ENGINEERING
# ============================================
def safe_divide(a, b, eps=1e-8):
    return a / (b + eps)

def compute_physics_features(df):
    out = df.copy()

    out["ptj1"] = np.sqrt(out["pxj1"]**2 + out["pyj1"]**2)
    out["ptj2"] = np.sqrt(out["pxj2"]**2 + out["pyj2"]**2)

    out["Ej1"] = np.sqrt(out["pxj1"]**2 + out["pyj1"]**2 + out["pzj1"]**2 + out["mj1"]**2)
    out["Ej2"] = np.sqrt(out["pxj2"]**2 + out["pyj2"]**2 + out["pzj2"]**2 + out["mj2"]**2)

    out["pxjj"] = out["pxj1"] + out["pxj2"]
    out["pyjj"] = out["pyj1"] + out["pyj2"]
    out["pzjj"] = out["pzj1"] + out["pzj2"]
    out["Ejj"]  = out["Ej1"] + out["Ej2"]

    mjj2 = out["Ejj"]**2 - out["pxjj"]**2 - out["pyjj"]**2 - out["pzjj"]**2
    mjj2 = np.maximum(mjj2, 0.0)
    out["mjj"] = np.sqrt(mjj2)

    out["tau21_j1"] = safe_divide(out["tau2j1"], out["tau1j1"])
    out["tau32_j1"] = safe_divide(out["tau3j1"], out["tau2j1"])
    out["tau21_j2"] = safe_divide(out["tau2j2"], out["tau1j2"])
    out["tau32_j2"] = safe_divide(out["tau3j2"], out["tau2j2"])

    out["pt_balance"] = np.abs(out["ptj1"] - out["ptj2"]) / (out["ptj1"] + out["ptj2"] + 1e-8)
    out["m_ratio"] = out["mj1"] / (out["mj2"] + 1e-8)

    return out

df_feat = compute_physics_features(df)
print("Feature-engineered dataframe shape:", df_feat.shape)
display(df_feat.head())


In [ ]:
# ============================================
# SECTION 9: PHYSICS DISTRIBUTIONS
# ============================================
plt.figure(figsize=(8, 5))
plt.hist(df_feat["mjj"], bins=150)
plt.xlabel("mjj")
plt.ylabel("Events")
plt.title("Dijet Invariant Mass Distribution")
plt.grid(alpha=0.3)
plt.show()

if "label" in df_feat.columns:
    plt.figure(figsize=(8, 5))
    plt.hist(df_feat[df_feat["label"] == 0]["mjj"], bins=150, alpha=0.6, label="Background")
    plt.hist(df_feat[df_feat["label"] == 1]["mjj"], bins=150, alpha=0.6, label="Signal")
    plt.xlabel("mjj")
    plt.ylabel("Events")
    plt.title("mjj Distribution by Truth Label")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

plot_cols = ["mj1", "mj2", "ptj1", "ptj2", "tau21_j1", "tau21_j2"]
for col in plot_cols:
    plt.figure(figsize=(7, 4))
    if "label" in df_feat.columns:
        plt.hist(df_feat[df_feat["label"] == 0][col], bins=100, alpha=0.6, density=True, label="Background")
        plt.hist(df_feat[df_feat["label"] == 1][col], bins=100, alpha=0.6, density=True, label="Signal")
        plt.legend()
    else:
        plt.hist(df_feat[col], bins=100, alpha=0.8, density=True)
    plt.xlabel(col)
    plt.ylabel("Density")
    plt.title(f"Distribution of {col}")
    plt.grid(alpha=0.3)
    plt.show()


## CWoLa region definition

Adjust the windows below after looking at the `mjj` plot.  
The current defaults target the resonance neighborhood around **3.5 TeV**.


In [ ]:
# ============================================
# SECTION 10: DEFINE SIGNAL REGION AND SIDEBANDS
# ============================================
SR_LOW = 3300
SR_HIGH = 3700

SB_LEFT_LOW = 2300
SB_LEFT_HIGH = 3000

SB_RIGHT_LOW = 4000
SB_RIGHT_HIGH = 4700

sr_mask = (df_feat["mjj"] >= SR_LOW) & (df_feat["mjj"] <= SR_HIGH)
sb_mask = (
    ((df_feat["mjj"] >= SB_LEFT_LOW) & (df_feat["mjj"] <= SB_LEFT_HIGH)) |
    ((df_feat["mjj"] >= SB_RIGHT_LOW) & (df_feat["mjj"] <= SB_RIGHT_HIGH))
)

df_sr = df_feat[sr_mask].copy()
df_sb = df_feat[sb_mask].copy()

print("Signal region events:", len(df_sr))
print("Sideband events     :", len(df_sb))

if "label" in df_feat.columns:
    print("\nTruth label fractions:")
    print("SR signal fraction:", df_sr["label"].mean())
    print("SB signal fraction:", df_sb["label"].mean())

plt.figure(figsize=(8,5))
plt.hist(df_feat["mjj"], bins=120, alpha=0.35, label="All events")
plt.axvspan(SR_LOW, SR_HIGH, alpha=0.25, label="Signal Region")
plt.axvspan(SB_LEFT_LOW, SB_LEFT_HIGH, alpha=0.18, label="Left Sideband")
plt.axvspan(SB_RIGHT_LOW, SB_RIGHT_HIGH, alpha=0.18, label="Right Sideband")
plt.xlabel("mjj")
plt.ylabel("Events")
plt.title("CWoLa Region Definition")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## Build the CWoLa dataset

Pseudo-labels:
- **1** for signal-region events
- **0** for sideband events

We exclude `mjj` from the training features to prevent trivial mass learning.


In [ ]:
# ============================================
# SECTION 11: BUILD THE CWOLA DATASET
# ============================================
df_sr["cwola_target"] = 1
df_sb["cwola_target"] = 0

cwola_df = pd.concat([df_sr, df_sb], axis=0).sample(frac=1.0, random_state=42).reset_index(drop=True)

feature_cols = [
    "ptj1", "ptj2",
    "mj1", "mj2",
    "tau21_j1", "tau32_j1",
    "tau21_j2", "tau32_j2",
    "pt_balance", "m_ratio",
    "pxj1", "pyj1", "pzj1",
    "pxj2", "pyj2", "pzj2"
]

X = cwola_df[feature_cols].copy()
y_cwola = cwola_df["cwola_target"].values
y_truth = cwola_df["label"].values if "label" in cwola_df.columns else None
mjj_vals = cwola_df["mjj"].values

print("Training matrix shape:", X.shape)
print("Pseudo-label distribution:", np.bincount(y_cwola))
if y_truth is not None:
    print("Truth-label distribution :", np.bincount(y_truth.astype(int)))


In [ ]:
# ============================================
# SECTION 12: TRAIN / VAL / TEST SPLIT
# ============================================
if y_truth is None:
    raise ValueError("Truth labels are not available in the dataframe.")

X_train, X_temp, y_train, y_temp, truth_train, truth_temp, mjj_train, mjj_temp = train_test_split(
    X, y_cwola, y_truth, mjj_vals,
    test_size=0.30,
    random_state=42,
    stratify=y_cwola
)

X_val, X_test, y_val, y_test, truth_val, truth_test, mjj_val, mjj_test = train_test_split(
    X_temp, y_temp, truth_temp, mjj_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Train:", X_train.shape)
print("Val  :", X_val.shape)
print("Test :", X_test.shape)


In [ ]:
# ============================================
# SECTION 13: SCALE FEATURES
# ============================================
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

print("Scaling complete.")


## CWoLa classifier

A dense neural classifier is used as a serious baseline for structured high-level features.


In [ ]:
# ============================================
# SECTION 14: BUILD THE MODEL
# ============================================
def build_cwola_model(input_dim):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(128, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(0.25),

        layers.Dense(64, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(0.20),

        layers.Dense(32, activation="relu"),
        layers.Dropout(0.10),

        layers.Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=["accuracy", keras.metrics.AUC(name="auc")]
    )
    return model

model = build_cwola_model(X_train_scaled.shape[1])
model.summary()


In [ ]:
# ============================================
# SECTION 15: CLASS WEIGHTS
# ============================================
classes = np.unique(y_train)
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weight = {int(c): float(w) for c, w in zip(classes, weights)}

print("Class weights:", class_weight)


In [ ]:
# ============================================
# SECTION 16: TRAINING
# ============================================
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_auc",
        mode="max",
        patience=8,
        restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=4,
        verbose=1
    )
]

history = model.fit(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=40,
    batch_size=2048,
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1
)


In [ ]:
# ============================================
# SECTION 17: TRAINING CURVES
# ============================================
hist = pd.DataFrame(history.history)
display(hist.head())

plt.figure(figsize=(8,5))
plt.plot(hist["loss"], label="Train Loss")
plt.plot(hist["val_loss"], label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

plt.figure(figsize=(8,5))
plt.plot(hist["auc"], label="Train AUC")
plt.plot(hist["val_auc"], label="Val AUC")
plt.xlabel("Epoch")
plt.ylabel("AUC")
plt.title("Training AUC")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

hist.to_csv(TABLE_DIR / "training_history.csv", index=False)


In [ ]:
# ============================================
# SECTION 18: PREDICT CWoLa SCORES
# ============================================
cwola_score_train = model.predict(X_train_scaled, batch_size=4096).ravel()
cwola_score_val   = model.predict(X_val_scaled, batch_size=4096).ravel()
cwola_score_test  = model.predict(X_test_scaled, batch_size=4096).ravel()

print("Prediction complete.")


In [ ]:
# ============================================
# SECTION 19: PSEUDO-LABEL EVALUATION
# ============================================
pseudo_auc = roc_auc_score(y_test, cwola_score_test)
pseudo_pred = (cwola_score_test >= 0.5).astype(int)
pseudo_acc = accuracy_score(y_test, pseudo_pred)

print("Pseudo-label Test AUC :", pseudo_auc)
print("Pseudo-label Test Acc :", pseudo_acc)

fpr, tpr, _ = roc_curve(y_test, cwola_score_test)

plt.figure(figsize=(6,6))
plt.plot(fpr, tpr, label=f"CWoLa pseudo-label ROC (AUC={pseudo_auc:.4f})")
plt.plot([0,1], [0,1], "--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Pseudo-label ROC")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

pd.DataFrame({"fpr": fpr, "tpr": tpr}).to_csv(TABLE_DIR / "pseudo_label_roc.csv", index=False)


In [ ]:
# ============================================
# SECTION 20: TRUTH-LABEL EVALUATION
# ============================================
truth_auc = roc_auc_score(truth_test, cwola_score_test)
print("Truth-label Test AUC:", truth_auc)

fpr_truth, tpr_truth, _ = roc_curve(truth_test, cwola_score_test)

plt.figure(figsize=(6,6))
plt.plot(fpr_truth, tpr_truth, label=f"Truth ROC (AUC={truth_auc:.4f})")
plt.plot([0,1], [0,1], "--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Truth-label ROC using CWoLa score")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

pd.DataFrame({"fpr": fpr_truth, "tpr": tpr_truth}).to_csv(TABLE_DIR / "truth_label_roc.csv", index=False)


In [ ]:
# ============================================
# SECTION 21: SCORE DISTRIBUTIONS
# ============================================
plt.figure(figsize=(8,5))
plt.hist(cwola_score_test[truth_test == 0], bins=80, alpha=0.6, density=True, label="Background")
plt.hist(cwola_score_test[truth_test == 1], bins=80, alpha=0.6, density=True, label="Signal")
plt.xlabel("CWoLa Score")
plt.ylabel("Density")
plt.title("CWoLa Score Distribution by Truth Label")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

score_dist_df = pd.DataFrame({
    "cwola_score": cwola_score_test,
    "truth_label": truth_test
})
score_dist_df.to_csv(TABLE_DIR / "score_distribution_data.csv", index=False)


In [ ]:
# ============================================
# SECTION 22: SCORE VS DIJET MASS
# ============================================
plt.figure(figsize=(8,5))
mask_bg = truth_test == 0
mask_sig = truth_test == 1
plt.scatter(mjj_test[mask_bg], cwola_score_test[mask_bg], s=4, alpha=0.12, label="Background")
plt.scatter(mjj_test[mask_sig], cwola_score_test[mask_sig], s=6, alpha=0.25, label="Signal")
plt.xlabel("mjj")
plt.ylabel("CWoLa Score")
plt.title("CWoLa Score vs Dijet Mass")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

pd.DataFrame({
    "mjj": mjj_test,
    "cwola_score": cwola_score_test,
    "truth_label": truth_test
}).to_csv(TABLE_DIR / "score_vs_mjj_data.csv", index=False)


In [ ]:
# ============================================
# SECTION 23: SIGNAL ENRICHMENT TABLE
# ============================================
test_results = pd.DataFrame({
    "mjj": mjj_test,
    "truth_label": truth_test,
    "cwola_score": cwola_score_test
})

rows = []
for q in [0.00, 0.90, 0.95, 0.99]:
    if q == 0.00:
        selected = test_results.copy()
        name = "All events"
    else:
        thr = np.quantile(cwola_score_test, q)
        selected = test_results[test_results["cwola_score"] >= thr]
        name = f"Top {int((1-q)*100)}%"
    signal_frac = selected["truth_label"].mean()
    rows.append({
        "Selection Region": name,
        "Events Selected": len(selected),
        "Signal Fraction": signal_frac
    })

enrichment_df = pd.DataFrame(rows)
display(enrichment_df)
enrichment_df.to_csv(TABLE_DIR / "signal_enrichment_table.csv", index=False)


In [ ]:
# ============================================
# SECTION 24: POST-SELECTION MASS DISTRIBUTION
# ============================================
thr_95 = np.quantile(cwola_score_test, 0.95)
selected_95 = test_results[test_results["cwola_score"] >= thr_95]

plt.figure(figsize=(8,5))
plt.hist(test_results["mjj"], bins=90, alpha=0.35, density=True, label="All test events")
plt.hist(selected_95["mjj"], bins=60, alpha=0.60, density=True, label="Top 5% score selection")
plt.xlabel("mjj")
plt.ylabel("Density")
plt.title("Mass Distribution After High-score Selection")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

selected_95.to_csv(TABLE_DIR / "postselection_mjj_data.csv", index=False)


In [ ]:
# ============================================
# SECTION 25: TOP CANDIDATE EVENTS
# ============================================
X_test_df = pd.DataFrame(X_test, columns=feature_cols).reset_index(drop=True)

candidates = X_test_df.copy()
candidates["mjj"] = mjj_test
candidates["truth_label"] = truth_test
candidates["cwola_score"] = cwola_score_test

top_candidates = candidates.sort_values("cwola_score", ascending=False).head(20).copy()
top_candidates.insert(0, "rank", np.arange(1, len(top_candidates) + 1))

display(top_candidates)
top_candidates.to_csv(TABLE_DIR / "top_candidate_events.csv", index=False)


In [ ]:
# ============================================
# SECTION 26: RESULTS SUMMARY TABLE
# ============================================
best_epoch = int(np.argmax(hist["val_auc"].values) + 1)

summary_table = pd.DataFrame({
    "Metric": [
        "Best Epoch",
        "Training Loss",
        "Validation Loss",
        "Training AUC",
        "Validation AUC",
        "Pseudo-label Test AUC",
        "Pseudo-label Test Accuracy",
        "Truth-label Test AUC"
    ],
    "Value": [
        best_epoch,
        float(hist.loc[best_epoch - 1, "loss"]),
        float(hist.loc[best_epoch - 1, "val_loss"]),
        float(hist.loc[best_epoch - 1, "auc"]),
        float(hist.loc[best_epoch - 1, "val_auc"]),
        float(pseudo_auc),
        float(pseudo_acc),
        float(truth_auc)
    ]
})

display(summary_table)
summary_table.to_csv(TABLE_DIR / "results_summary_table.csv", index=False)


In [ ]:
# ============================================
# SECTION 27: SAVE MODEL AND ENGINEERED FEATURES
# ============================================
ENGINEERED_PATH = PROCESSED_DIR / "lhco_engineered_features.parquet"
df_feat.to_parquet(ENGINEERED_PATH, index=False)

MODEL_PATH = MODEL_DIR / "cwola_classifier.keras"
model.save(MODEL_PATH)

print("Saved engineered features to:", ENGINEERED_PATH)
print("Saved model to:", MODEL_PATH)
